# Market Structures

This notebook walks through the `market_structures` package, which provides objects for representing market data.

The package contains two sub-packages:
- **`interpolation`** — 1-D interpolators: `LinearInterpolator`, `LogLinearInterpolator`, `V2TInterpolator`
- **`rates`** — `ZeroCurve`: an interpolated zero-rate curve supporting discount factor, zero rate, and forward rate queries

## Environment setup

The cell below calls `setup_demo_env()` from `examples/_setup.py`, which performs three steps:

1. Adds the project root to `sys.path` so first-party packages (`database`, `schedules`, `market_structures`, `credit`, ...) import without installation.
2. Redirects the global SQLite path to `examples/demo.db` via `set_db_path()`. The production `quant.db` is never read or written by this notebook.
3. Creates the domain tables and seeds the holidays table (USD/EUR/GBP/PLN, years 2000-2100) on first run; subsequent runs detect existing data and skip seeding.

A status line is printed when the cell runs so you can see which path was taken.

In [ ]:
import sys
sys.path.insert(0, "..")

from examples._setup import setup_demo_env
setup_demo_env()

In [ ]:
from datetime import date
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from market_structures import (
    ZeroCurve,
    LinearInterpolator,
    LogLinearInterpolator,
    V2TInterpolator,
)
from market_conventions import DayCountConvention, CompoundingType

## 1. Interpolators

All interpolators share the same interface: given sorted pillar `xs` and corresponding `ys`, compute `y` at any point `x`.

Outside the pillar range, flat extrapolation is applied by default.

| Interpolator | Interpolates on | Designed for |
|---|---|---|
| `LinearInterpolator` | y directly | General use |
| `LogLinearInterpolator` | log(y) | Discount factors (market standard) |
| `V2TInterpolator` | total variance σ²·T | Implied volatility surfaces |

In [ ]:
# Pillar times (years) and discount factors
xs = [0.5, 1.0, 2.0, 5.0]
ys = [0.978, 0.955, 0.910, 0.800]

x_query = 1.5  # midpoint between 1Y and 2Y pillars

linear = LinearInterpolator()
loglin = LogLinearInterpolator()

print(f"Query point:              t = {x_query}")
print(f"LinearInterpolator:       {linear.interpolate(x_query, xs, ys):.6f}")
print(f"LogLinearInterpolator:    {loglin.interpolate(x_query, xs, ys):.6f}")
print()
print("Log-linear gives a slightly lower value because it interpolates on log(y),")
print("which accounts for the convexity of discount factor decay.")

### V2TInterpolator

`V2TInterpolator` is designed for implied volatility surfaces. It interpolates by linearly blending **total variance** (σ²·T) across expiries, then converting back to volatility. This ensures total variance is non-decreasing — a necessary condition to prevent calendar spread arbitrage.

In [ ]:
# Pillar expiries (years) and implied vols
vol_xs = [0.25, 0.5, 1.0, 2.0]
vol_ys = [0.20, 0.22, 0.25, 0.28]

v2t = V2TInterpolator()

expiries = [0.25, 0.375, 0.5, 0.75, 1.0, 1.5, 2.0]
vols = [v2t.interpolate(t, vol_xs, vol_ys) for t in expiries]
total_variances = [v**2 * t for v, t in zip(vols, expiries)]

print(f"{'Expiry (T)':<14} {'Implied Vol':<14} {'Total Variance (σ²·T)'}")
print("-" * 45)
for t, v, tv in zip(expiries, vols, total_variances):
    print(f"{t:<14.3f} {v:<14.4f} {tv:.6f}")
print("\nTotal variance is non-decreasing — no calendar spread arbitrage.")

## 2. ZeroCurve

A `ZeroCurve` is constructed from:
- A **reference date** — the curve's anchor point (today)
- **Pillar dates** and corresponding **zero rates**
- A **day count convention** — used to compute time fractions
- A **compounding type** — how rates are expressed (default: continuous)
- An **interpolator** — how rates/discount factors are estimated between pillars (default: log-linear)

Internally, the curve converts zero rates to discount factors at construction time and interpolates on discount factors.

In [ ]:
# Build a humped yield curve — rates rise then fall at the long end.
# This shape makes interpolation differences more visible.

REF = date(2024, 1, 1)

def pillar(months):
    return REF + relativedelta(months=months)

pillar_dates = [
    pillar(1),   # 1M
    pillar(3),   # 3M
    pillar(6),   # 6M
    pillar(9),   # 9M
    pillar(12),  # 1Y
    pillar(18),  # 18M
    pillar(24),  # 2Y
    pillar(36),  # 3Y
    pillar(60),  # 5Y
    pillar(84),  # 7Y
    pillar(120), # 10Y
]

rates = [
    0.0450,  # 1M
    0.0460,  # 3M
    0.0475,  # 6M
    0.0485,  # 9M
    0.0490,  # 1Y
    0.0500,  # 18M  <- peak
    0.0505,  # 2Y
    0.0500,  # 3Y
    0.0480,  # 5Y
    0.0460,  # 7Y
    0.0440,  # 10Y
]

curve = ZeroCurve(
    reference_date=REF,
    pillar_dates=pillar_dates,
    rates=rates,
    day_count_convention=DayCountConvention.ACT_365_FIXED,
    compounding_type=CompoundingType.CONTINUOUS,
)

print(f"{'Pillar':<12} {'Zero Rate':>12} {'Discount Factor':>18}")
print("-" * 44)
labels = ['1M','3M','6M','9M','1Y','18M','2Y','3Y','5Y','7Y','10Y']
for label, d, r in zip(labels, pillar_dates, rates):
    df = curve.discount_factor(d)
    print(f"{label:<12} {r:>11.2%}  {df:>18.6f}")

### Zero Rates at Non-Pillar Dates

The curve can return a zero rate at any date, not just the pillars. The interpolator fills in the gaps.

In [ ]:
query_dates = [
    ("2M",  pillar(2)),
    ("4M",  pillar(4)),
    ("15M", pillar(15)),
    ("4Y",  pillar(48)),
    ("8Y",  pillar(96)),
]

print(f"{'Tenor':<8} {'Date':<14} {'Zero Rate':>12} {'Discount Factor':>18}")
print("-" * 55)
for label, d in query_dates:
    zr = curve.zero_rate(d)
    df = curve.discount_factor(d)
    print(f"{label:<8} {str(d):<14} {zr:>11.4%}  {df:>18.6f}")

### Pillar Management

Pillars can be added or removed dynamically. The curve maintains sorted order internally.

In [ ]:
# Add a 4Y pillar
new_pillar = pillar(48)
before = curve.zero_rate(new_pillar)

curve.add_pillar(new_pillar, 0.0492)
after = curve.zero_rate(new_pillar)

print(f"Zero rate at 4Y before adding pillar: {before:.4%}")
print(f"Zero rate at 4Y after  adding pillar: {after:.4%}  (now exactly 4.92%)")

# Remove it again
curve.remove_pillar(new_pillar)
restored = curve.zero_rate(new_pillar)
print(f"Zero rate at 4Y after removing pillar: {restored:.4%}  (back to interpolated value)")

## 3. Effect of Interpolation on Forward Rates

A **forward rate** is the rate implied by the curve for a future period. It is computed from the ratio of two discount factors:

$$f(t_1, t_2) = -\frac{\ln(P(t_2) / P(t_1))}{t_2 - t_1}$$

Forward rates are highly sensitive to how discount factors are interpolated between pillars — even small differences in interpolation produce visible differences in the forward rate curve.

Below we compare **linear** and **log-linear** interpolation by computing rolling 6-month forward rates across the curve.

In [ ]:
curve_linear = ZeroCurve(
    reference_date=REF,
    pillar_dates=pillar_dates,
    rates=rates,
    day_count_convention=DayCountConvention.ACT_365_FIXED,
    compounding_type=CompoundingType.CONTINUOUS,
    interpolator=LinearInterpolator(),
)

curve_loglin = ZeroCurve(
    reference_date=REF,
    pillar_dates=pillar_dates,
    rates=rates,
    day_count_convention=DayCountConvention.ACT_365_FIXED,
    compounding_type=CompoundingType.CONTINUOUS,
    interpolator=LogLinearInterpolator(),
)

# Compute 6-month forward rates at monthly steps from 1M to 9Y6M
fwd_starts = [pillar(m) for m in range(1, 115)]
fwd_ends   = [pillar(m + 6) for m in range(1, 115)]

fwd_linear = [curve_linear.forward_rate(s, e) for s, e in zip(fwd_starts, fwd_ends)]
fwd_loglin = [curve_loglin.forward_rate(s, e) for s, e in zip(fwd_starts, fwd_ends)]

# Time axis in years from reference
t_axis = [(s - REF).days / 365.25 for s in fwd_starts]

# Mark pillar positions
pillar_t = [(d - REF).days / 365.25 for d in pillar_dates]
pillar_fwd_loglin = [curve_loglin.forward_rate(d, d + relativedelta(months=6)) for d in pillar_dates[:-1]]

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(t_axis, [r * 100 for r in fwd_loglin], label='Log-Linear (market standard)', color='steelblue', linewidth=2)
ax.plot(t_axis, [r * 100 for r in fwd_linear], label='Linear', color='tomato', linewidth=2, linestyle='--')
ax.scatter(pillar_t, [r * 100 for r in rates], zorder=5, color='black', s=40, label='Pillar zero rates')

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel('Start of forward period (years)')
ax.set_ylabel('6-month forward rate')
ax.set_title('6-Month Forward Rates: Linear vs Log-Linear Interpolation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify the difference between the two methods
diffs_bps = [(l - ll) * 10000 for l, ll in zip(fwd_linear, fwd_loglin)]

print("Difference in 6M forward rates (Linear minus Log-Linear), in basis points:")
print(f"  Max absolute difference: {max(abs(d) for d in diffs_bps):.2f} bps")
print(f"  Mean absolute difference: {sum(abs(d) for d in diffs_bps) / len(diffs_bps):.2f} bps")
print()
print("Linear interpolation produces 'kinked' forward rates — they change slope abruptly")
print("at each pillar. Log-linear interpolation is smoother and is the market standard")
print("because it reflects the multiplicative nature of discount factor compounding.")